# Clustering d'articles PREVYO à partir des sous-graphes d'événements

Ce notebook construit un **clustering au niveau article** en utilisant `resultAnalyseId` pour regrouper tous les sous-graphes appartenant au même article.

## Idée
Chaque article devient un vecteur construit à partir de :

- `type`, `domain`, `subdomain`, `risk`, `riskCarac`
- les **labels** des nœuds
- l'**arborescence complète** des labels, par exemple `Thing`, `Thing/Abstract`, `Thing/Abstract/Event`, `Thing/Abstract/Event/Win`
- le couple **form + label**
- les propriétés linguistiques des nœuds comme `mood`, `aspect`, `category`, `tense`, `polarity`

## Entrée attendue
Le notebook attend le fichier :

```text
data/export_event.json
```

## Sorties générées
- `data/articles_clusters.json` pour la visualisation
- `app_cluster_d3.html` pour explorer les clusters en D3


In [3]:
from __future__ import annotations

import json
import math
import os
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics import silhouette_score


## 1. Paramètres

Tu peux modifier ces paramètres si besoin.  
Par défaut, le notebook :

- regroupe par `resultAnalyseId`
- garde l'arborescence complète des labels
- utilise KMeans sur une représentation TF-IDF
- choisit automatiquement le meilleur `k` avec le score de silhouette


In [ ]:
DATA_PATH = Path("data/export.events.json")
OUTPUT_JSON = Path("data/articles_clusters.json")
OUTPUT_HTML = Path("app_cluster_d3.html")

RANDOM_STATE = 42
MAX_K = 12
KEEP_FORM_TOKENS = True
KEEP_FORM_LABEL_TOKENS = True
KEEP_EDGE_TYPES = False
MIN_DOC_FREQUENCY = 2
MAX_DOC_FREQUENCY_RATIO = 0.95

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)


## 2. Chargement du JSON

Ce chargeur gère :

- un JSON classique sous forme de liste
- un fichier JSON Lines si jamais l'export est ligne par ligne


In [5]:
def load_json_events(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(
            f"Fichier introuvable : {path}\n"
            "Place ton export dans data/export_event.json puis relance le notebook."
        )

    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Le fichier {path} est vide.")

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        raise ValueError("Le JSON principal n'est pas une liste.")
    except Exception:
        events = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            events.append(json.loads(line))
        return events


events = load_json_events(DATA_PATH)
print(f"Nombre de sous-graphes chargés : {len(events):,}".replace(",", " "))


Nombre de sous-graphes chargés : 11 948


## 3. Aperçu rapide des données

In [6]:
preview_cols = ["resultAnalyseId", "type", "domain", "subdomain", "risk", "riskCarac", "context"]
preview_df = pd.DataFrame([{k: e.get(k) for k in preview_cols} for e in events])
display(preview_df.head(5))


,resultAnalyseId,type,domain,subdomain,risk,riskCarac,context
0,698b35e042a8083a290c11c7,Thing/Abstract/Event/Win,NaN,Thing/Abstract/Domain/Digital,Thing/Abstract/Risk/Societal,Thing/Abstract/Domain/Sovereignty,Portrait des six principaux candidats qui sont...
1,698b35e042a8083a290c11c5,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Thing/Abstract/Event/Spying,NaN,NaN,NaN,Des e-mails publiés par le département américa...
2,698b35e042a8083a290c11c5,Thing/Abstract/Event/Transfer/TransferOfUnbias...,Thing/Abstract/Event/Spying,NaN,NaN,NaN,Des e-mails publiés par le département américa...
3,698b35e042a8083a290c11c5,Thing/Abstract/Event/Build,Thing/Abstract/Event/Spying,NaN,NaN,NaN,ervi de facilitateur à des représentants de l’...
4,698b35e042a8083a290c11c5,Thing/Abstract/Event/Addition,Thing/Abstract/Event/Spying,NaN,NaN,NaN,Des e-mails publiés par le département américa...


## 4. Fonctions de transformation

### Ce qu'on fabrique
On transforme chaque sous-graphe en ensemble de **tokens sémantiques**.

Exemple pour un label :

```text
Thing/Abstract/Event/Win
```

On génère les niveaux :

```text
Thing
Thing/Abstract
Thing/Abstract/Event
Thing/Abstract/Event/Win
```

Cela permet de capturer à la fois :
- la catégorie générale
- la spécialisation fine


In [7]:
def normalize_text(value: object) -> str:
    if value is None:
        return ""
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return value


def hierarchy_tokens(path: str | None, prefix: str) -> list[str]:
    if not path or not isinstance(path, str):
        return []
    parts = [part for part in path.split("/") if part]
    tokens = []
    for i in range(1, len(parts) + 1):
        tokens.append(f"{prefix}::{'/'.join(parts[:i])}")
    return tokens


def stringify_property_value(value: object) -> str:
    if value is None:
        return "NULL"
    if isinstance(value, list):
        return "|".join(map(str, value))
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False, sort_keys=True)
    return str(value)


def extract_event_tokens(event: dict) -> Counter:
    tokens = Counter()

    # Champs de haut niveau demandés dans la consigne
    for field in ["type", "domain", "subdomain", "risk", "riskCarac"]:
        for token in hierarchy_tokens(event.get(field), field):
            tokens[token] += 1

    # Nœuds
    for node in event.get("nodes") or []:
        form = normalize_text(node.get("form"))

        if KEEP_FORM_TOKENS and form:
            tokens[f"form::{form}"] += 1

        for label in node.get("labels") or []:
            for token in hierarchy_tokens(label, "node_label"):
                tokens[token] += 1

            if KEEP_FORM_LABEL_TOKENS and form and label:
                tokens[f"form_label::{form}__{label}"] += 1

        for prop_name, prop_value in (node.get("properties") or {}).items():
            prop_value = stringify_property_value(prop_value)
            tokens[f"prop::{prop_name}={prop_value}"] += 1

    # Arêtes optionnelles
    if KEEP_EDGE_TYPES:
        for edge in event.get("edges") or []:
            edge_type = edge.get("type")
            if edge_type:
                tokens[f"edge::{edge_type}"] += 1

    return tokens


## 5. Agrégation au niveau article avec `resultAnalyseId`

C'est ici qu'on applique la logique importante de ton besoin :

- plusieurs sous-graphes peuvent appartenir au même article
- `resultAnalyseId` sert de clé d'agrégation
- on additionne donc tous les labels et propriétés des sous-graphes d'un même article


In [8]:
articles = defaultdict(lambda: {
    "resultAnalyseId": None,
    "tokens": Counter(),
    "n_events": 0,
    "contexts": [],
    "types": set(),
    "domains": set(),
    "subdomains": set(),
    "risks": set(),
    "riskCaracs": set(),
})

for event in events:
    article_id = event.get("resultAnalyseId") or "UNKNOWN_RESULT_ANALYSE_ID"
    article = articles[article_id]

    article["resultAnalyseId"] = article_id
    article["tokens"].update(extract_event_tokens(event))
    article["n_events"] += 1

    context = event.get("context")
    if context:
        article["contexts"].append(context)

    if event.get("type"):
        article["types"].add(event["type"])
    if event.get("domain"):
        article["domains"].add(event["domain"])
    if event.get("subdomain"):
        article["subdomains"].add(event["subdomain"])
    if event.get("risk"):
        article["risks"].add(event["risk"])
    if event.get("riskCarac"):
        article["riskCaracs"].add(event["riskCarac"])

article_records = list(articles.values())
print(f"Nombre d'articles reconstruits : {len(article_records):,}".replace(",", " "))


Nombre d'articles reconstruits : 2 273


## 6. Contrôle rapide après agrégation

In [9]:
check_df = pd.DataFrame([
    {
        "resultAnalyseId": a["resultAnalyseId"],
        "n_events": a["n_events"],
        "n_unique_tokens": len(a["tokens"]),
        "sample_context": (a["contexts"][0][:120] + "...") if a["contexts"] else ""
    }
    for a in article_records
]).sort_values(["n_events", "n_unique_tokens"], ascending=False)

display(check_df.head(10))


,resultAnalyseId,n_events,n_unique_tokens,sample_context
507,698dc8a9c87e717556cdcc14,21,180,"Le choix de ""Parlons-en"" aujourd’hui porte sur..."
1774,699a2ae8bf23573ae1689d1b,19,184,"À 24 ans, Mialitiana Clerc dispute ses troisiè..."
1522,699849e3bf23573ae1689c1c,19,155,Le président américain Donald Trump a annoncé ...
1121,69930e9f7eb36c285af0a9e1,18,169,"SOS, les sols s’épuisent L’agriculture mondial..."
756,698f5aca7eb36c285af0a86b,18,161,Beaucoup de spécialistes alertent sur les risq...
860,69905a897eb36c285af0a8d6,17,121,"Cette semaine, dans le podcast Avec Judith, l’..."
293,698c67e0a75d7e3055e288ce,16,169,En persistant à arborer un casque honorant plu...
1546,69986ad2bf23573ae1689c32,16,150,Des centaines de Kenyans ont été enrôlés pour ...
1361,6997016fbf23573ae1689b6e,16,128,Le réchauffement climatique augmente le risque...
1507,6998397abf23573ae1689c0a,15,196,Le président américain Donald Trump a dit jeud...


## 7. Filtrage des tokens trop rares ou trop fréquents

On enlève :

- les tokens trop rares, souvent très bruités
- les tokens présents dans presque tous les articles, souvent peu discriminants

Cela améliore la qualité du clustering.


In [10]:
doc_frequency = Counter()
for article in article_records:
    doc_frequency.update(article["tokens"].keys())

n_articles = len(article_records)

effective_min_df = 1 if n_articles < 30 else MIN_DOC_FREQUENCY

def keep_token(token: str) -> bool:
    df = doc_frequency[token]
    if df < effective_min_df:
        return False
    if n_articles > 1 and (df / n_articles) > MAX_DOC_FREQUENCY_RATIO:
        return False
    return True

filtered_feature_dicts = []
article_ids = []

for article in article_records:
    filtered_tokens = {
        token: value
        for token, value in article["tokens"].items()
        if keep_token(token)
    }
    filtered_feature_dicts.append(filtered_tokens)
    article_ids.append(article["resultAnalyseId"])

print(f"Nombre moyen de tokens conservés par article : {np.mean([len(x) for x in filtered_feature_dicts]):.2f}")


Nombre moyen de tokens conservés par article : 56.75


## 8. Vectorisation TF-IDF

In [11]:
vectorizer = DictVectorizer()
X_counts = vectorizer.fit_transform(filtered_feature_dicts)

tfidf = TfidfTransformer()
X = tfidf.fit_transform(X_counts)

print("Matrice articles x features :", X.shape)
print("Nombre de features conservées :", len(vectorizer.feature_names_))


Matrice articles x features : (2273, 9529)
Nombre de features conservées : 9529


## 9. Choix automatique du nombre de clusters

On teste plusieurs valeurs de `k` puis on garde celle qui donne le meilleur score de silhouette.  
Si le jeu est trop petit, on force une solution simple.


In [12]:
def choose_k(X, max_k: int = 12, random_state: int = 42) -> tuple[int, dict]:
    n_samples = X.shape[0]

    if n_samples <= 2:
        return 1, {}

    upper = min(max_k, n_samples - 1)
    candidate_ks = list(range(2, upper + 1))

    scores = {}
    for k in candidate_ks:
        model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
        labels = model.fit_predict(X)
        if len(set(labels)) < 2:
            continue
        try:
            scores[k] = silhouette_score(X, labels)
        except Exception:
            continue

    if not scores:
        return 2, {}

    best_k = max(scores, key=scores.get)
    return best_k, scores


best_k, silhouette_scores = choose_k(X, max_k=MAX_K, random_state=RANDOM_STATE)
print("k retenu :", best_k)
print("scores silhouette :", silhouette_scores)


k retenu : 11
scores silhouette : {2: 0.026169015856493713, 3: 0.03208384638179682, 4: 0.03674281274256066, 5: 0.043252301730376176, 6: 0.048704212422167, 7: 0.045386867252949596, 8: 0.046685553257401737, 9: 0.047998825294746694, 10: 0.05397761801674629, 11: 0.059076042084244815, 12: 0.056743975769785036}


## 10. Clustering final

In [13]:
if X.shape[0] == 1:
    cluster_labels = np.array([0], dtype=int)
    kmeans = None
else:
    n_clusters = max(1, min(best_k, X.shape[0]))
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=20)
    cluster_labels = kmeans.fit_predict(X)

cluster_series = pd.Series(cluster_labels, name="cluster")
display(cluster_series.value_counts().sort_index().rename_axis("cluster").to_frame("n_articles"))


,n_articles
cluster,
0,261
1,166
2,57
3,215
4,767
5,132
6,138
7,100
8,208


## 11. Projection 2D pour la visualisation D3

On calcule ici des coordonnées 2D pour chaque article.

Le clustering se fait dans l'espace TF-IDF complet.  
La projection 2D sert seulement à l'exploration visuelle.


In [14]:
def project_to_2d(X) -> np.ndarray:
    n_samples, n_features = X.shape

    if n_samples == 1:
        return np.array([[0.0, 0.0]])

    if n_features == 0:
        return np.zeros((n_samples, 2))

    if n_features == 1:
        x = X.toarray().ravel()
        return np.column_stack([x, np.zeros_like(x)])

    n_components = 2 if n_features >= 2 else 1
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    coords = svd.fit_transform(X)

    if coords.shape[1] == 1:
        coords = np.column_stack([coords[:, 0], np.zeros(coords.shape[0])])

    return coords


coords = project_to_2d(X)
coords[:5]


array([[ 0.2618652 , -0.05465292],
       [ 0.24976948,  0.01813624],
       [ 0.15977262, -0.0562481 ],
       [ 0.45785062, -0.00094086],
       [ 0.24607656, -0.27979359]])

## 12. Interprétation des clusters

Pour chaque cluster, on extrait quelques features les plus caractéristiques.


In [15]:
feature_names = np.array(vectorizer.feature_names_)

def top_cluster_features(X, labels, feature_names, top_n=12):
    summaries = []
    X_dense = X.toarray()

    global_mean = X_dense.mean(axis=0)

    for cluster_id in sorted(set(labels)):
        mask = labels == cluster_id
        cluster_mean = X_dense[mask].mean(axis=0)
        lift = cluster_mean - global_mean
        top_idx = np.argsort(lift)[::-1][:top_n]

        summaries.append({
            "cluster": int(cluster_id),
            "size": int(mask.sum()),
            "top_features": [str(feature_names[i]) for i in top_idx if cluster_mean[i] > 0]
        })

    return summaries


cluster_summaries = top_cluster_features(X, cluster_labels, feature_names, top_n=12)
pd.DataFrame(cluster_summaries)


,cluster,size,top_features
0,0,261,"[subdomain::Thing/Abstract/Domain/Digital, sub..."
1,1,166,[risk::Thing/Abstract/Risk/EconomicIntelligenc...
2,2,57,"[subdomain::Thing/Abstract/Domain/Nuclear, dom..."
3,3,215,"[riskCarac::Thing/Abstract/Event, riskCarac::T..."
4,4,767,"[node_label::Thing/Concrete, node_label::Thing..."
5,5,132,"[risk::Thing/Abstract/Risk/Societal, risk::Thi..."
6,6,138,[domain::Thing/Abstract/Organization/Operator/...
7,7,100,"[risk::Thing/Abstract/Risk/Cyber, domain::Thin..."
8,8,208,"[domain::Thing/Abstract/Organization, domain::..."
9,9,147,"[risk::Thing/Abstract/Risk/Natural, riskCarac:..."


## 13. Préparation de l'export JSON pour D3

In [16]:
def humanize_feature(token: str) -> str:
    if "::" not in token:
        return token
    prefix, value = token.split("::", 1)

    labels = {
        "type": "type",
        "domain": "domain",
        "subdomain": "subdomain",
        "risk": "risk",
        "riskCarac": "riskCarac",
        "node_label": "label",
        "form": "forme",
        "form_label": "forme + label",
        "prop": "propriété",
        "edge": "arête",
    }
    return f"{labels.get(prefix, prefix)}: {value}"


def article_top_features(token_counter: Counter, kept_tokens: set[str], top_n: int = 15) -> list[str]:
    items = [(k, v) for k, v in token_counter.items() if k in kept_tokens]
    items = sorted(items, key=lambda x: (-x[1], x[0]))[:top_n]
    return [humanize_feature(k) for k, _ in items]


def make_excerpt(text: str | None, max_len: int = 220) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) <= max_len:
        return text
    return text[: max_len - 1].rstrip() + "…"


def unique_preserve_order(values: list[str]) -> list[str]:
    seen = set()
    out = []
    for value in values:
        key = str(value).strip()
        if not key or key in seen:
            continue
        seen.add(key)
        out.append(value)
    return out


kept_token_set = set(vectorizer.feature_names_)

articles_export = []
for idx, article in enumerate(article_records):
    context_excerpts = [make_excerpt(c) for c in unique_preserve_order(article["contexts"])[:5]]
    articles_export.append({
        "resultAnalyseId": article["resultAnalyseId"],
        "cluster": int(cluster_labels[idx]),
        "x": float(coords[idx, 0]),
        "y": float(coords[idx, 1]),
        "n_events": int(article["n_events"]),
        "n_unique_tokens": int(len(article["tokens"])),
        "types": sorted(article["types"]),
        "domains": sorted(article["domains"]),
        "subdomains": sorted(article["subdomains"]),
        "risks": sorted(article["risks"]),
        "riskCaracs": sorted(article["riskCaracs"]),
        "contexts": article["contexts"][:5],
        "context_excerpts": context_excerpts,
        "top_features": article_top_features(article["tokens"], kept_token_set, top_n=15),
    })

clusters_export = []
for cluster_summary in cluster_summaries:
    cluster_id = cluster_summary["cluster"]
    cluster_member_articles = [
        article
        for i, article in enumerate(article_records)
        if int(cluster_labels[i]) == cluster_id
    ]

    member_ids = [article["resultAnalyseId"] for article in cluster_member_articles][:10]

    example_contexts = []
    for article in cluster_member_articles:
        unique_contexts = unique_preserve_order(article["contexts"])
        if not unique_contexts:
            continue
        example_contexts.append({
            "resultAnalyseId": article["resultAnalyseId"],
            "excerpt": make_excerpt(unique_contexts[0]),
        })
        if len(example_contexts) >= 6:
            break

    clusters_export.append({
        "cluster": int(cluster_id),
        "size": int(cluster_summary["size"]),
        "top_features": [humanize_feature(f) for f in cluster_summary["top_features"]],
        "example_resultAnalyseIds": member_ids,
        "example_contexts": example_contexts,
    })

export_payload = {
    "meta": {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "source_file": str(DATA_PATH),
        "n_subgraphs": int(len(events)),
        "n_articles": int(len(article_records)),
        "n_features": int(len(vectorizer.feature_names_)),
        "n_clusters": int(len(set(cluster_labels))),
        "method": "Article-level clustering from aggregated event subgraphs using resultAnalyseId + TF-IDF + KMeans"
    },
    "clusters": clusters_export,
    "articles": articles_export
}

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(export_payload, f, ensure_ascii=False, indent=2)

print(f"Export JSON écrit dans : {OUTPUT_JSON}")


C:\Users\antoc\AppData\Local\Temp\ipykernel_34268\173917000.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


Export JSON écrit dans : data\articles.clusters.json


## 14. Génération de la page HTML D3

La page lit `data/articles_clusters.json`, puis affiche :

- un nuage 2D des articles
- une légende des clusters
- une fiche de détail
- une recherche par `resultAnalyseId`
- un filtre par cluster


In [17]:
HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="fr">
<head>
  <meta charset="UTF-8" />
  <title>PREVYO - Clustering des articles</title>
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
  <style>
    :root {
      --bg: #0f172a;
      --panel: #111827;
      --panel-2: #1f2937;
      --text: #e5e7eb;
      --muted: #94a3b8;
      --border: #334155;
      --accent: #38bdf8;
      --ok: #10b981;
      --danger: #f87171;
    }

    * { box-sizing: border-box; }

    body {
      margin: 0;
      font-family: Inter, Arial, sans-serif;
      background: linear-gradient(180deg, #0b1220 0%, #0f172a 100%);
      color: var(--text);
    }

    header {
      padding: 20px 24px 12px 24px;
      border-bottom: 1px solid var(--border);
      background: rgba(15, 23, 42, 0.92);
      position: sticky;
      top: 0;
      z-index: 5;
      backdrop-filter: blur(8px);
    }

    header h1 {
      margin: 0 0 6px 0;
      font-size: 24px;
    }

    header p {
      margin: 0;
      color: var(--muted);
      line-height: 1.4;
    }

    .layout {
      display: grid;
      grid-template-columns: 340px minmax(600px, 1fr) 360px;
      gap: 16px;
      padding: 16px;
    }

    .panel {
      background: rgba(17, 24, 39, 0.9);
      border: 1px solid var(--border);
      border-radius: 16px;
      padding: 14px;
      min-height: 120px;
      box-shadow: 0 10px 30px rgba(0,0,0,0.2);
    }

    .panel h2, .panel h3 {
      margin-top: 0;
    }

    .meta-grid {
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 10px;
      margin-top: 12px;
    }

    .meta-card {
      border: 1px solid var(--border);
      background: var(--panel-2);
      border-radius: 12px;
      padding: 10px;
    }

    .meta-card .label {
      color: var(--muted);
      font-size: 12px;
      margin-bottom: 4px;
    }

    .meta-card .value {
      font-size: 18px;
      font-weight: 700;
    }

    .control-group {
      margin-bottom: 14px;
    }

    .control-group label {
      display: block;
      font-size: 13px;
      color: var(--muted);
      margin-bottom: 6px;
    }

    input, select, button {
      width: 100%;
      border-radius: 10px;
      border: 1px solid var(--border);
      background: #0b1220;
      color: var(--text);
      padding: 10px 12px;
      font-size: 14px;
    }

    button {
      cursor: pointer;
      transition: transform .08s ease, background .2s ease;
    }

    button:hover {
      transform: translateY(-1px);
      background: #162033;
    }

    .legend-item {
      display: grid;
      grid-template-columns: 18px 1fr;
      gap: 10px;
      align-items: start;
      padding: 10px 0;
      border-bottom: 1px solid rgba(51, 65, 85, 0.45);
    }

    .legend-color {
      width: 14px;
      height: 14px;
      border-radius: 999px;
      margin-top: 4px;
    }

    .small {
      color: var(--muted);
      font-size: 12px;
      line-height: 1.5;
    }

    #chart-wrap {
      position: relative;
      min-height: 720px;
    }

    #chart {
      width: 100%;
      height: 720px;
      display: block;
      border-radius: 14px;
      background: radial-gradient(circle at top, rgba(56, 189, 248, 0.08), transparent 35%), rgba(15, 23, 42, 0.88);
    }

    .tooltip {
      position: absolute;
      pointer-events: none;
      background: rgba(2, 6, 23, 0.96);
      border: 1px solid var(--border);
      border-radius: 12px;
      padding: 10px 12px;
      font-size: 12px;
      max-width: 300px;
      opacity: 0;
      transition: opacity .12s ease;
      box-shadow: 0 8px 26px rgba(0,0,0,0.35);
    }

    .pill {
      display: inline-block;
      padding: 4px 8px;
      border-radius: 999px;
      background: rgba(56, 189, 248, 0.12);
      border: 1px solid rgba(56, 189, 248, 0.25);
      color: #bae6fd;
      font-size: 12px;
      margin: 0 6px 6px 0;
    }

    .feature-list, .context-list {
      margin: 0;
      padding-left: 18px;
      line-height: 1.5;
    }

    .context-list li {
      margin-bottom: 8px;
    }

    .empty {
      color: var(--muted);
      font-style: italic;
    }

    .point {
      stroke: rgba(255,255,255,0.9);
      stroke-width: 1.1px;
      cursor: pointer;
    }

    .point.dimmed {
      opacity: 0.12;
    }

    .point.active {
      stroke: #ffffff;
      stroke-width: 2.5px;
    }

    .axis text, .axis path, .axis line {
      color: var(--muted);
      stroke: var(--muted);
      font-size: 12px;
    }

    .grid line {
      stroke: rgba(148, 163, 184, 0.14);
    }

    .grid path {
      stroke: none;
    }

    .footer-note {
      color: var(--muted);
      font-size: 12px;
      margin-top: 10px;
      line-height: 1.5;
    }

    @media (max-width: 1400px) {
      .layout {
        grid-template-columns: 1fr;
      }
      #chart {
        height: 640px;
      }
    }
  </style>
</head>
<body>
  <header>
    <h1>PREVYO - Clustering d'articles par agrégation de sous-graphes</h1>
    <p>
      Les articles sont reconstruits via <strong>resultAnalyseId</strong>, puis clusterisés à partir de
      <strong>type</strong>, <strong>domain</strong>, <strong>subdomain</strong>, <strong>risk</strong>,
      <strong>riskCarac</strong>, des <strong>labels hiérarchiques</strong>, des couples <strong>form + label</strong>
      et des <strong>propriétés linguistiques</strong>.
    </p>
  </header>

  <main class="layout">
    <section class="panel">
      <h2>Contrôles</h2>

      <div class="control-group">
        <label for="search">Recherche par resultAnalyseId</label>
        <input id="search" type="text" placeholder="Ex. 698b35e042a8083a290c11c5" />
      </div>

      <div class="control-group">
        <label for="clusterFilter">Filtre cluster</label>
        <select id="clusterFilter">
          <option value="all">Tous les clusters</option>
        </select>
      </div>

      <div class="control-group">
        <button id="resetBtn">Réinitialiser la sélection</button>
      </div>

      <div id="meta" class="meta-grid"></div>

      <div class="footer-note">
        La taille des points dépend du nombre de sous-graphes agrégés dans l'article.
        La couleur représente le cluster.
      </div>
    </section>

    <section class="panel">
      <h2>Projection 2D des articles</h2>
      <div id="chart-wrap">
        <svg id="chart"></svg>
        <div id="tooltip" class="tooltip"></div>
      </div>
      <div class="footer-note">
        Les coordonnées 2D servent à l'exploration visuelle.
        Le clustering lui-même a été calculé dans l'espace TF-IDF complet.
      </div>
    </section>

    <section class="panel">
      <h2>Détail</h2>
      <div id="detail">
        <p class="empty">Clique sur un point pour afficher le détail d'un article.</p>
      </div>

      <h3 style="margin-top:20px;">Résumé des clusters</h3>
      <div id="legend"></div>
    </section>
  </main>

  <script>
    const DATA_URL = "data/articles_clusters.json";

    const svg = d3.select("#chart");
    const tooltip = d3.select("#tooltip");
    const detail = d3.select("#detail");
    const meta = d3.select("#meta");
    const legend = d3.select("#legend");
    const searchInput = document.getElementById("search");
    const clusterFilter = document.getElementById("clusterFilter");
    const resetBtn = document.getElementById("resetBtn");

    const width = () => document.getElementById("chart").clientWidth || 900;
    const height = () => document.getElementById("chart").clientHeight || 720;
    const margin = { top: 24, right: 24, bottom: 48, left: 52 };

    let state = {
      data: null,
      selectedId: null,
      search: "",
      cluster: "all"
    };

    function renderMeta(metaData) {
      const cards = [
        { label: "Sous-graphes", value: metaData.n_subgraphs },
        { label: "Articles", value: metaData.n_articles },
        { label: "Features", value: metaData.n_features },
        { label: "Clusters", value: metaData.n_clusters }
      ];

      meta.selectAll("*").remove();

      meta.selectAll("div")
        .data(cards)
        .join("div")
        .attr("class", "meta-card")
        .html(d => `
          <div class="label">${d.label}</div>
          <div class="value">${d.value}</div>
        `);
    }

    function renderLegend(clusters, color) {
      legend.selectAll("*").remove();

      const items = legend.selectAll(".legend-item")
        .data(clusters)
        .join("div")
        .attr("class", "legend-item");

      items.html(d => `
        <div class="legend-color" style="background:${color(d.cluster)}"></div>
        <div>
          <div><strong>Cluster ${d.cluster}</strong> · ${d.size} article(s)</div>
          <div class="small" style="margin-top:6px;">
            ${(d.top_features || []).slice(0, 6).map(f => `<span class="pill">${f}</span>`).join("")}
          </div>
          <div class="small" style="margin-top:6px;">
            Exemples : ${(d.example_resultAnalyseIds || []).slice(0, 4).join(", ")}
          </div>
          <div style="margin-top:8px;">
            <div class="small">Extraits de contexte de resultAnalyseId différents</div>
            <ul class="context-list">
              ${(d.example_contexts || []).slice(0, 3).map(item => `<li><strong>${item.resultAnalyseId}</strong> · ${item.excerpt}</li>`).join("") || "<li class='empty'>Aucun extrait</li>"}
            </ul>
          </div>
        </div>
      `);
    }

    function renderDetail(article, color) {
      if (!article) {
        detail.html(`<p class="empty">Clique sur un point pour afficher le détail d'un article.</p>`);
        return;
      }

      const tagBlock = (values) => (values || []).map(v => `<span class="pill">${v}</span>`).join("");
      const clusterSummary = (state.data?.clusters || []).find(c => String(c.cluster) === String(article.cluster));
      const otherClusterContexts = (clusterSummary?.example_contexts || [])
        .filter(item => String(item.resultAnalyseId) !== String(article.resultAnalyseId))
        .slice(0, 4);

      detail.html(`
        <div style="display:flex;align-items:center;gap:10px;margin-bottom:12px;">
          <div class="legend-color" style="background:${color(article.cluster)};width:16px;height:16px;"></div>
          <div><strong>${article.resultAnalyseId}</strong></div>
        </div>

        <div class="small" style="margin-bottom:10px;">
          Cluster ${article.cluster} · ${article.n_events} sous-graphe(s) agrégé(s) · ${article.n_unique_tokens} token(s)
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">Types</div>
          <div>${tagBlock(article.types) || '<span class="empty">Aucun</span>'}</div>
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">Domaines</div>
          <div>${tagBlock(article.domains) || '<span class="empty">Aucun</span>'}</div>
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">Subdomains</div>
          <div>${tagBlock(article.subdomains) || '<span class="empty">Aucun</span>'}</div>
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">Risques</div>
          <div>${tagBlock(article.risks) || '<span class="empty">Aucun</span>'}</div>
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">RiskCarac</div>
          <div>${tagBlock(article.riskCaracs) || '<span class="empty">Aucun</span>'}</div>
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">Top features</div>
          <ul class="feature-list">
            ${(article.top_features || []).slice(0, 12).map(f => `<li>${f}</li>`).join("") || "<li class='empty'>Aucune</li>"}
          </ul>
        </div>

        <div style="margin-bottom:12px;">
          <div class="small">Extraits de contexte de cet article</div>
          <ul class="context-list">
            ${(article.context_excerpts || article.contexts || []).slice(0, 4).map(c => `<li>${c}</li>`).join("") || "<li class='empty'>Aucun extrait</li>"}
          </ul>
        </div>

        <div>
          <div class="small">Extraits de contexte de resultAnalyseId différents dans le même cluster</div>
          <ul class="context-list">
            ${otherClusterContexts.map(item => `<li><strong>${item.resultAnalyseId}</strong> · ${item.excerpt}</li>`).join("") || "<li class='empty'>Aucun extrait dans ce cluster</li>"}
          </ul>
        </div>
      `);
    }

    function getVisibleArticles(articles) {
      const search = state.search.trim().toLowerCase();
      return articles.filter(a => {
        const okCluster = state.cluster === "all" || String(a.cluster) === String(state.cluster);
        const okSearch = !search || String(a.resultAnalyseId).toLowerCase().includes(search);
        return okCluster && okSearch;
      });
    }

    function updateClusterOptions(clusters) {
      clusterFilter.innerHTML = `<option value="all">Tous les clusters</option>` +
        clusters.map(c => `<option value="${c.cluster}">Cluster ${c.cluster}</option>`).join("");
    }

    function renderChart(data) {
      const allArticles = data.articles || [];
      const visibleArticles = getVisibleArticles(allArticles);

      svg.selectAll("*").remove();

      const W = width();
      const H = height();
      svg.attr("viewBox", `0 0 ${W} ${H}`);

      const root = svg.append("g");

      const clusterIds = [...new Set((data.clusters || []).map(d => d.cluster))].sort((a, b) => a - b);
      const color = d3.scaleOrdinal()
        .domain(clusterIds)
        .range(d3.schemeTableau10.concat(d3.schemeSet3).slice(0, clusterIds.length));

      renderLegend(data.clusters || [], color);

      if (!visibleArticles.length) {
        root.append("text")
          .attr("x", W / 2)
          .attr("y", H / 2)
          .attr("text-anchor", "middle")
          .attr("fill", "#94a3b8")
          .text("Aucun article ne correspond au filtre.");
        renderDetail(null, color);
        return;
      }

      const xExtent = d3.extent(allArticles, d => d.x);
      const yExtent = d3.extent(allArticles, d => d.y);

      const x = d3.scaleLinear()
        .domain(d3.extent([
          xExtent[0] - 0.1 * (xExtent[1] - xExtent[0] || 1),
          xExtent[1] + 0.1 * (xExtent[1] - xExtent[0] || 1)
        ]))
        .range([margin.left, W - margin.right]);

      const y = d3.scaleLinear()
        .domain(d3.extent([
          yExtent[0] - 0.1 * (yExtent[1] - yExtent[0] || 1),
          yExtent[1] + 0.1 * (yExtent[1] - yExtent[0] || 1)
        ]))
        .range([H - margin.bottom, margin.top]);

      const radius = d3.scaleSqrt()
        .domain(d3.extent(allArticles, d => d.n_events))
        .range([5, 16]);

      const xGrid = d3.axisBottom(x).ticks(8).tickSize(-(H - margin.top - margin.bottom)).tickFormat("");
      const yGrid = d3.axisLeft(y).ticks(8).tickSize(-(W - margin.left - margin.right)).tickFormat("");

      root.append("g")
        .attr("class", "grid")
        .attr("transform", `translate(0, ${H - margin.bottom})`)
        .call(xGrid);

      root.append("g")
        .attr("class", "grid")
        .attr("transform", `translate(${margin.left}, 0)`)
        .call(yGrid);

      root.append("g")
        .attr("class", "axis")
        .attr("transform", `translate(0, ${H - margin.bottom})`)
        .call(d3.axisBottom(x).ticks(8));

      root.append("g")
        .attr("class", "axis")
        .attr("transform", `translate(${margin.left}, 0)`)
        .call(d3.axisLeft(y).ticks(8));

      root.append("text")
        .attr("x", W / 2)
        .attr("y", H - 10)
        .attr("fill", "#94a3b8")
        .attr("text-anchor", "middle")
        .text("Dimension 1");

      root.append("text")
        .attr("transform", `translate(16, ${H / 2}) rotate(-90)`)
        .attr("fill", "#94a3b8")
        .attr("text-anchor", "middle")
        .text("Dimension 2");

      const points = root.append("g")
        .selectAll("circle")
        .data(allArticles, d => d.resultAnalyseId)
        .join("circle")
        .attr("class", "point")
        .attr("cx", d => x(d.x))
        .attr("cy", d => y(d.y))
        .attr("r", d => radius(d.n_events))
        .attr("fill", d => color(d.cluster))
        .classed("dimmed", d => !visibleArticles.some(v => v.resultAnalyseId === d.resultAnalyseId))
        .classed("active", d => state.selectedId === d.resultAnalyseId)
        .on("mouseenter", function(event, d) {
          tooltip.style("opacity", 1)
            .html(`
              <div><strong>${d.resultAnalyseId}</strong></div>
              <div>Cluster ${d.cluster}</div>
              <div>${d.n_events} sous-graphe(s)</div>
              <div style="margin-top:6px;" class="small">${(d.top_features || []).slice(0, 4).join("<br>")}</div>
            `);
        })
        .on("mousemove", function(event) {
          const wrap = document.getElementById("chart-wrap").getBoundingClientRect();
          tooltip
            .style("left", (event.clientX - wrap.left + 12) + "px")
            .style("top", (event.clientY - wrap.top + 12) + "px");
        })
        .on("mouseleave", function() {
          tooltip.style("opacity", 0);
        })
        .on("click", function(event, d) {
          state.selectedId = d.resultAnalyseId;
          renderAll();
        });

      const selectedArticle = allArticles.find(a => a.resultAnalyseId === state.selectedId);
      renderDetail(selectedArticle || visibleArticles[0] || null, color);
    }

    function renderAll() {
      renderChart(state.data);
    }

    d3.json(DATA_URL).then(data => {
      state.data = data;
      renderMeta(data.meta || {});
      updateClusterOptions(data.clusters || []);
      renderAll();
    }).catch(err => {
      document.body.innerHTML = `
        <div style="padding:28px;font-family:Arial,sans-serif;color:#e5e7eb;background:#0f172a;min-height:100vh;">
          <h1>Impossible de charger data/articles_clusters.json</h1>
          <p>Erreur : ${String(err)}</p>
          <p>Exécute d'abord le notebook pour générer le fichier JSON de sortie.</p>
        </div>
      `;
    });

    searchInput.addEventListener("input", e => {
      state.search = e.target.value || "";
      renderAll();
    });

    clusterFilter.addEventListener("change", e => {
      state.cluster = e.target.value;
      renderAll();
    });

    resetBtn.addEventListener("click", () => {
      state.selectedId = null;
      state.search = "";
      state.cluster = "all";
      searchInput.value = "";
      clusterFilter.value = "all";
      renderAll();
    });

    window.addEventListener("resize", () => {
      if (state.data) {
        renderAll();
      }
    });
  </script>
</body>
</html>
'''

with OUTPUT_HTML.open("w", encoding="utf-8") as f:
    f.write(HTML_TEMPLATE)

print(f"Page HTML écrite dans : {OUTPUT_HTML}")


Page HTML écrite dans : app_cluster_d3.html


## 15. Vérification finale des sorties

In [18]:
display(pd.DataFrame(articles_export).head(5))
print("Fichiers générés :")
print("-", OUTPUT_JSON.resolve())
print("-", OUTPUT_HTML.resolve())


,resultAnalyseId,cluster,x,y,n_events,n_unique_tokens,types,domains,subdomains,risks,riskCaracs,contexts,context_excerpts,top_features
0,698b35e042a8083a290c11c7,10,0.261865,-0.054653,1,35,[Thing/Abstract/Event/Win],[],[Thing/Abstract/Domain/Digital],[Thing/Abstract/Risk/Societal],[Thing/Abstract/Domain/Sovereignty],[Portrait des six principaux candidats qui son...,[Portrait des six principaux candidats qui son...,"[forme: 15et 22mars, forme: fauteuil, forme: r..."
1,698b35e042a8083a290c11c5,4,0.249769,0.018136,4,54,"[Thing/Abstract/Event/Addition, Thing/Abstract...",[Thing/Abstract/Event/Spying],[],[],[],[Des e-mails publiés par le département améric...,[Des e-mails publiés par le département améric...,"[domain: Thing, domain: Thing/Abstract, domain..."
2,698b35e042a8083a290c11cc,4,0.159773,-0.056248,6,70,"[Thing/Abstract/Event/Communication, Thing/Abs...",[],[],[],[],"[Le procureur a tenu une conférence de presse,...","[Le procureur a tenu une conférence de presse,...","[label: Thing/Concrete, label: Thing/Concrete/..."
3,698b35ea42a8083a290c11cf,8,0.457851,-0.000941,5,71,"[Thing/Abstract/Event/Addition, Thing/Abstract...",[Thing/Abstract/Organization/Operator/Civil],[],[Thing/Abstract/Risk/EconomicIntelligence],[],[Par une série de nouvelles mesures techniques...,[Par une série de nouvelles mesures techniques...,"[domain: Thing, domain: Thing/Abstract, domain..."
4,698b35e042a8083a290c11c6,3,0.246077,-0.279794,4,65,"[Thing/Abstract/Event/Addition, Thing/Abstract...",[],[],[],[Thing/Abstract/Event/Destabilization],[Près de 6000 clients étaient sans électricité...,[Près de 6000 clients étaient sans électricité...,"[propriété: mood=EMPTY, propriété: tense=EMPTY..."


Fichiers générés :
- C:\Users\antoc\PREVYO_Marathon_du_web\data\articles.clusters.json
- C:\Users\antoc\PREVYO_Marathon_du_web\app_cluster_d3.html


## 16. Comment utiliser le résultat

### Dans Jupyter
1. Place ton export dans `data/export_event.json`
2. Exécute toutes les cellules
3. Ouvre ensuite `app_cluster_d3.html`

### En serveur local
Pour que le navigateur charge correctement les fichiers JSON, lance un petit serveur local à la racine du projet :

```bash
python -m http.server 8000
```

Puis ouvre :

```text
http://localhost:8000/app_cluster_d3.html
```

### Ce que montre la visualisation
- chaque point = un article reconstruit par `resultAnalyseId`
- la couleur = le cluster
- la taille = le nombre de sous-graphes agrégés
- le panneau de droite = les types, risques, contextes, top features de l'article et des extraits d'autres resultAnalyseId du même cluster
